# SIM2 angle/colour cosine comparisons

This notebook mirrors the SIM 2 image perturbation figures by focusing on the four Jeep paint colours and the Wolf reference rendered at 45° increments. We compute CLIP embeddings to compare how colour and pose influence similarity, first deterministically and then under Monte Carlo dropout (`p=0.01`, 32 passes).

In [8]:
%pip install -r ../requirements.txt
%pip install -e ..
print('Environment bootstrap complete. Installed requirements and editable package.')

Note: you may need to restart the kernel to use updated packages.
Obtaining file:///home/fwromano/Documents/Code/UCLIP
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for uclip (pyproject.toml) ... done
  Created wheel for uclip: filename=uclip-0.1.0-0.editable-py3-none-any.whl size=2852 sha256=e0622e8ec946151587f2c1f80d67e0e0ea274041a86cb547b6e3bb121f688506
  Stored in directory: /tmp/pip-ephem-wheel-cache-333ohstz/wheels/26/1e/ad/e88c9da78e5477b133fdc3a114e0c0028513e6e0f31e4d0b56
Successfully built uclip
  Attempting uninstall: uclip
    Found existing installation: uclip 0.1.0
    Uninstalling uclip-0.1.0:
      Successfully uninstalled uclip-0.1.0
Note: you may need to restart the kernel to use updated packages.
Environment bootstrap complete. Installed requirements and editable package.


In [1]:
import sys
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
import hashlib

NOTEBOOK_ROOT = Path.cwd()
REPO_ROOT = NOTEBOOK_ROOT if (NOTEBOOK_ROOT / 'src').exists() else NOTEBOOK_ROOT.parent
SRC_ROOT = REPO_ROOT / 'src'
for path in (REPO_ROOT, SRC_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

pd.set_option('display.precision', 4)
print(f'Repository root: {REPO_ROOT}')


Repository root: /home/fwromano/Documents/Code/UCLIP


In [2]:
DATA_ROOT = REPO_ROOT / 'data/car_sim/sim2_cropped_45deg'
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset not found at {DATA_ROOT}')

TARGET_COLOURS = ['Indigo', 'Magenta', 'White', 'Yellow', 'Wolf2']
DISPLAY_NAMES = {'Wolf2': 'Wolf'}

def _parse_angle(path: Path) -> int:
    token = path.stem.split('_')[-1]
    try:
        return int(token)
    except ValueError as exc:
        raise ValueError(f'Could not parse angle from {path.name}') from exc

records = []
for path in sorted(DATA_ROOT.glob('*/*.png')):
    colour = path.parent.name
    if colour not in TARGET_COLOURS:
        continue
    angle = _parse_angle(path)
    records.append({
        'color': colour,
        'display_color': DISPLAY_NAMES.get(colour, colour),
        'angle': angle,
        'path': path,
    })

dataset_df = pd.DataFrame(records).sort_values(['color', 'angle']).reset_index(drop=True)
angles = sorted(dataset_df['angle'].unique().tolist())
colours = [colour for colour in TARGET_COLOURS if colour in dataset_df['color'].unique()]
print(f'Loaded {len(dataset_df)} images spanning {len(colours)} colours and {len(angles)} angles.')
display(dataset_df.head())


CACHE_ROOT = REPO_ROOT / 'runs' / 'sim2_angle_colour_cosine'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)


Loaded 40 images spanning 5 colours and 8 angles.


,color,display_color,angle,path
0,Indigo,Indigo,0,/home/fwromano/Documents/Code/UCLIP/data/car_s...
1,Indigo,Indigo,45,/home/fwromano/Documents/Code/UCLIP/data/car_s...
2,Indigo,Indigo,90,/home/fwromano/Documents/Code/UCLIP/data/car_s...
3,Indigo,Indigo,135,/home/fwromano/Documents/Code/UCLIP/data/car_s...
4,Indigo,Indigo,180,/home/fwromano/Documents/Code/UCLIP/data/car_s...


## Helper utilities

In [3]:
from uclip.core.utils import load_clip_backbone, normalise_embeddings, set_determinism
from uclip.core.dropout import insert_adapters, override_dropout_rate, dump_dropout_rates
from uclip.core.sampling import sample_embeddings

CACHE_ROOT.mkdir(parents=True, exist_ok=True)

def _cache_key(colour: str, angle: int) -> str:
    return f"{colour}::{angle}"

def load_cached_embeddings(path: Path):
    if not path.exists():
        return None
    raw = torch.load(path, map_location='cpu')
    embeddings = {}
    for key, tensor in raw.items():
        if isinstance(key, str):
            colour, angle_str = key.split('::', 1)
            angle = int(angle_str)
        else:
            colour, angle = key
        embeddings[(colour, int(angle))] = tensor.clone().cpu()
    return embeddings

def save_cached_embeddings(path: Path, embeddings):
    payload = {_cache_key(colour, angle): tensor.clone().cpu() for (colour, angle), tensor in embeddings.items()}
    torch.save(payload, path)

def aggregate_vectors(vectors):
    stacked = torch.stack(vectors)
    mean = stacked.mean(dim=0, keepdim=True)
    return normalise_embeddings(mean).squeeze(0)

def compute_embeddings(df: pd.DataFrame, processor, device, generator) -> Dict[Tuple[str, int], torch.Tensor]:
    embeddings: Dict[Tuple[str, int], torch.Tensor] = {}
    for row in df.itertuples(index=False):
        with Image.open(row.path) as image:
            image = image.convert('RGB')
            inputs = processor(images=image, return_tensors='pt')
        inputs = {key: value.to(device) for key, value in inputs.items()}
        embedding = generator(inputs)
        embeddings[(row.color, row.angle)] = embedding.detach().cpu()
    return embeddings

def compute_variant_embeddings(df: pd.DataFrame, processor, device, generator, transform_fn, repeats: int = 1) -> Dict[Tuple[str, int], torch.Tensor]:
    embeddings: Dict[Tuple[str, int], torch.Tensor] = {}
    for row in df.itertuples(index=False):
        with Image.open(row.path) as image:
            base = image.convert('RGB')
        vectors = []
        for rep in range(repeats):
            transformed = transform_fn(base, row, rep)
            inputs = processor(images=transformed, return_tensors='pt')
            inputs = {key: value.to(device) for key, value in inputs.items()}
            vector = generator(inputs)
            vectors.append(vector.detach().cpu())
        embeddings[(row.color, row.angle)] = aggregate_vectors(vectors)
    return embeddings

def build_similarity_tables(embeddings: Dict[Tuple[str, int], torch.Tensor], colour_order, angle_order, display_map):
    angle_tables = {}
    for angle in angle_order:
        vectors = torch.stack([embeddings[(colour, angle)] for colour in colour_order])
        matrix = (vectors @ vectors.T).cpu().numpy()
        labels = [display_map.get(colour, colour) for colour in colour_order]
        angle_tables[angle] = pd.DataFrame(matrix, index=labels, columns=labels)
    colour_tables = {}
    for colour in colour_order:
        vectors = torch.stack([embeddings[(colour, angle)] for angle in angle_order])
        matrix = (vectors @ vectors.T).cpu().numpy()
        colour_tables[colour] = pd.DataFrame(matrix, index=angle_order, columns=angle_order)
    return angle_tables, colour_tables

def ensure_cached_embeddings(cache_path: Path, builder):
    embeddings = load_cached_embeddings(cache_path)
    if embeddings is not None:
        return embeddings, False
    embeddings = builder()
    save_cached_embeddings(cache_path, embeddings)
    return embeddings, True


## Perturbation configuration

In [4]:
def derive_seed(*parts: object, offset: int = 0) -> int:
    token = '|'.join(str(part) for part in parts)
    digest = hashlib.sha256(token.encode('utf-8')).digest()
    base = int.from_bytes(digest[:8], 'little')
    return (base + offset) % (2**32 - 1)

def apply_gaussian_noise(image: Image.Image, sigma: float, seed: int) -> Image.Image:
    rng = np.random.default_rng(seed)
    arr = np.asarray(image).astype(np.float32) / 255.0
    noise = rng.normal(0.0, sigma, size=arr.shape)
    noisy = np.clip(arr + noise, 0.0, 1.0)
    return Image.fromarray((noisy * 255).round().astype(np.uint8))

def gaussian_transform_factory(sigma: float):
    def transform(image: Image.Image, row, replicate: int) -> Image.Image:
        seed = derive_seed('gaussian', row.color, row.angle, f'{sigma:.6f}', replicate)
        return apply_gaussian_noise(image, sigma, seed)
    return transform

def pixelate_transform(percent: int):
    scale = max(1.0 - percent / 100.0, 1e-4)
    def transform(image: Image.Image, _row, _rep) -> Image.Image:
        w, h = image.size
        target = (max(1, int(round(w * scale))), max(1, int(round(h * scale))))
        lowres = image.resize(target, resample=Image.NEAREST)
        return lowres.resize((w, h), resample=Image.NEAREST)
    return transform

def bicubic_transform(percent: int):
    scale = max(1.0 - percent / 100.0, 1e-4)
    def transform(image: Image.Image, _row, _rep) -> Image.Image:
        w, h = image.size
        target = (max(1, int(round(w * scale))), max(1, int(round(h * scale))))
        lowres = image.resize(target, resample=Image.BICUBIC)
        return lowres.resize((w, h), resample=Image.BICUBIC)
    return transform


def format_float_tag(value: float) -> str:
    text = f"{value:.6f}".rstrip('0').rstrip('.')
    return text.replace('.', 'p')

GAUSSIAN_SIGMAS = [0.01, 0.02, 0.04, 0.08, 0.16, 0.32]
GAUSSIAN_REPEATS = 16
DOWNSAMPLE_PERCENTS = [10, 20, 40, 60, 80]


## Deterministic CLIP embeddings

In [5]:
MODEL_ID = 'openai/clip-vit-base-patch32'
DEVICE = None  # set to 'cuda' to force GPU usage
SEED = 20240410

DET_CACHE_PATH = CACHE_ROOT / f"{MODEL_ID.replace('/', '_')}_deterministic.pt"

det_embeddings = load_cached_embeddings(DET_CACHE_PATH)
if det_embeddings is not None:
    print(f'Loaded deterministic embeddings from cache: {DET_CACHE_PATH}')
else:
    set_determinism(SEED)
    loaded = load_clip_backbone(MODEL_ID, device=DEVICE)
    model_det, processor_det, device_det = loaded.model, loaded.processor, loaded.device
    model_det.eval()

    def deterministic_generator(inputs):
        with torch.no_grad():
            embedding = model_det.get_image_features(**inputs)
        embedding = normalise_embeddings(embedding).squeeze(0)
        return embedding

    det_embeddings = compute_embeddings(dataset_df, processor_det, device_det, deterministic_generator)
    save_cached_embeddings(DET_CACHE_PATH, det_embeddings)
    print(f'Deterministic embeddings computed and cached at {DET_CACHE_PATH}')

if det_embeddings is None:
    raise RuntimeError('Deterministic embeddings unavailable after attempted computation.')

angle_tables_det, colour_tables_det = build_similarity_tables(det_embeddings, colours, angles, DISPLAY_NAMES)
print('Deterministic embeddings ready for similarity analysis.')


Loaded deterministic embeddings from cache: /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_deterministic.pt
Deterministic embeddings ready for similarity analysis.


In [6]:
det_tables = list(angle_tables_det.values()) + list(colour_tables_det.values())
if not det_tables:
    raise ValueError('No deterministic similarity tables available.')
det_vmin = min(table.values.min() for table in det_tables)
det_vmax = max(table.values.max() for table in det_tables)

for angle in angles:
    print(f'\nAngle {angle}° — cross-colour cosine similarity')
    display(
        angle_tables_det[angle]
        .style.format('{:.4f}')
        .background_gradient(cmap='viridis', vmin=det_vmin, vmax=det_vmax)
    )

print('\nDeterministic cosine similarity across angle pairs for each colour:')
for colour in colours:
    label = DISPLAY_NAMES.get(colour, colour)
    print(f'\n{label} — cross-angle cosine similarity')
    display(
        colour_tables_det[colour]
        .style.format('{:.4f}')
        .background_gradient(cmap='viridis', vmin=det_vmin, vmax=det_vmax)
    )



Angle 0° — cross-colour cosine similarity


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.8835,0.8654,0.8990,0.6927
Magenta,0.8835,1.0000,0.8698,0.8736,0.6650
White,0.8654,0.8698,1.0000,0.9027,0.7064
Yellow,0.8990,0.8736,0.9027,1.0000,0.7295
Wolf,0.6927,0.6650,0.7064,0.7295,1.0000



Angle 45° — cross-colour cosine similarity


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9130,0.8902,0.9038,0.6922
Magenta,0.9130,1.0000,0.8801,0.8883,0.6792
White,0.8902,0.8801,1.0000,0.9257,0.7659
Yellow,0.9038,0.8883,0.9257,1.0000,0.7333
Wolf,0.6922,0.6792,0.7659,0.7333,1.0000



Angle 90° — cross-colour cosine similarity


...

,0,45,90,135,180,225,270,315
0,1.0000,0.8399,0.8452,0.8205,0.8635,0.8155,0.8691,0.8423
45,0.8399,1.0000,0.8491,0.9005,0.7647,0.9040,0.8802,0.9674
90,0.8452,0.8491,1.0000,0.8423,0.8333,0.8440,0.9544,0.8494
135,0.8205,0.9005,0.8423,1.0000,0.8097,0.9438,0.8668,0.9025
180,0.8635,0.7647,0.8333,0.8097,1.0000,0.7950,0.8476,0.7680
225,0.8155,0.9040,0.8440,0.9438,0.7950,1.0000,0.8659,0.8969
270,0.8691,0.8802,0.9544,0.8668,0.8476,0.8659,1.0000,0.8880
315,0.8423,0.9674,0.8494,0.9025,0.7680,0.8969,0.8880,1.0000



Yellow — cross-angle cosine similarity


,0,45,90,135,180,225,270,315
0,1.0000,0.8777,0.8491,0.8564,0.9104,0.8092,0.8547,0.8559
45,0.8777,1.0000,0.8435,0.9315,0.8592,0.8961,0.8679,0.9531
90,0.8491,0.8435,1.0000,0.8481,0.8373,0.8327,0.9176,0.8436
135,0.8564,0.9315,0.8481,1.0000,0.8602,0.9267,0.8784,0.9302
180,0.9104,0.8592,0.8373,0.8602,1.0000,0.8207,0.8337,0.8376
225,0.8092,0.8961,0.8327,0.9267,0.8207,1.0000,0.8744,0.9142
270,0.8547,0.8679,0.9176,0.8784,0.8337,0.8744,1.0000,0.8801
315,0.8559,0.9531,0.8436,0.9302,0.8376,0.9142,0.8801,1.0000



Wolf — cross-angle cosine similarity


,0,45,90,135,180,225,270,315
0,1.0000,0.9000,0.8420,0.8577,0.9658,0.8878,0.8548,0.8978
45,0.9000,1.0000,0.8931,0.9428,0.8522,0.9441,0.8373,0.9668
90,0.8420,0.8931,1.0000,0.8814,0.8181,0.8831,0.9255,0.8873
135,0.8577,0.9428,0.8814,1.0000,0.8347,0.9579,0.8339,0.9397
180,0.9658,0.8522,0.8181,0.8347,1.0000,0.8673,0.8498,0.8493
225,0.8878,0.9441,0.8831,0.9579,0.8673,1.0000,0.8407,0.9582
270,0.8548,0.8373,0.9255,0.8339,0.8498,0.8407,1.0000,0.8496
315,0.8978,0.9668,0.8873,0.9397,0.8493,0.9582,0.8496,1.0000


## MC dropout (p=0.01, 32 passes)

In [7]:
MC_DROPOUT_P = 0.01
MC_PASSES = 32
MC_MICROBATCH = 4
ADAPTER_TARGETS = [f'vision_model.encoder.layers.{idx}' for idx in range(12)] + ['visual_projection']

mc_cache_name = f"{MODEL_ID.replace('/', '_')}_mcdo_p{MC_DROPOUT_P}_passes{MC_PASSES}.pt"
MC_CACHE_PATH = CACHE_ROOT / mc_cache_name

mcdo_embeddings = load_cached_embeddings(MC_CACHE_PATH)
if mcdo_embeddings is not None:
    print(f'Loaded MC dropout embeddings from cache: {MC_CACHE_PATH}')
else:
    set_determinism(SEED)
    loaded_mc = load_clip_backbone(MODEL_ID, device=DEVICE)
    model_mc, processor_mc, device_mc = loaded_mc.model, loaded_mc.processor, loaded_mc.device
    model_mc.eval()
    insert_adapters(model_mc, ADAPTER_TARGETS, p=MC_DROPOUT_P)
    override_dropout_rate(model_mc, MC_DROPOUT_P)
    dropout_count, dropout_rates = dump_dropout_rates(model_mc)
    if dropout_rates:
        print(f'Detected {dropout_count} dropout layers with rates between {min(dropout_rates):.4f} and {max(dropout_rates):.4f}.')
    else:
        print('No dropout layers detected; MC dropout adapters may not have been inserted as expected.')

    def mcdo_generator(inputs):
        samples = sample_embeddings(
            model=model_mc,
            forward_fn=model_mc.get_image_features,
            forward_kwargs=inputs,
            passes=MC_PASSES,
            microbatch=MC_MICROBATCH,
        )
        model_mc.eval()
        samples = normalise_embeddings(samples)
        mean_embed = samples.mean(dim=0, keepdim=True)
        mean_embed = normalise_embeddings(mean_embed).squeeze(0)
        return mean_embed

    mcdo_embeddings = compute_embeddings(dataset_df, processor_mc, device_mc, mcdo_generator)
    save_cached_embeddings(MC_CACHE_PATH, mcdo_embeddings)
    print(f'MC dropout embeddings computed and cached at {MC_CACHE_PATH}')

if mcdo_embeddings is None:
    raise RuntimeError('MC dropout embeddings unavailable after attempted computation.')

angle_tables_mcdo, colour_tables_mcdo = build_similarity_tables(mcdo_embeddings, colours, angles, DISPLAY_NAMES)
print('Monte Carlo dropout means ready for similarity analysis.')


Loaded MC dropout embeddings from cache: /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mcdo_p0.01_passes32.pt
Monte Carlo dropout means ready for similarity analysis.


In [8]:
mc_tables = list(angle_tables_mcdo.values()) + list(colour_tables_mcdo.values())
if not mc_tables:
    raise ValueError('No MC dropout similarity tables available.')
mc_vmin = min(table.values.min() for table in mc_tables)
mc_vmax = max(table.values.max() for table in mc_tables)

for angle in angles:
    print(f'\nAngle {angle}° — cross-colour cosine similarity (MC dropout mean)')
    display(
        angle_tables_mcdo[angle]
        .style.format('{:.4f}')
        .background_gradient(cmap='viridis', vmin=mc_vmin, vmax=mc_vmax)
    )

print('\nMonte Carlo dropout cosine similarity across angle pairs for each colour:')
for colour in colours:
    label = DISPLAY_NAMES.get(colour, colour)
    print(f'\n{label} — cross-angle cosine similarity (MC dropout mean)')
    display(
        colour_tables_mcdo[colour]
        .style.format('{:.4f}')
        .background_gradient(cmap='viridis', vmin=mc_vmin, vmax=mc_vmax)
    )



Angle 0° — cross-colour cosine similarity (MC dropout mean)


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9065,0.9015,0.9249,0.7488
Magenta,0.9065,1.0000,0.8790,0.8778,0.6968
White,0.9015,0.8790,1.0000,0.9164,0.7417
Yellow,0.9249,0.8778,0.9164,1.0000,0.7725
Wolf,0.7488,0.6968,0.7417,0.7725,1.0000



Angle 45° — cross-colour cosine similarity (MC dropout mean)


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9152,0.9077,0.9135,0.7461
Magenta,0.9152,1.0000,0.8892,0.8879,0.7263
White,0.9077,0.8892,1.0000,0.9314,0.8111
Yellow,0.9135,0.8879,0.9314,1.0000,0.7657
Wolf,0.7461,0.7263,0.8111,0.7657,1.0000



Angle 90° — cross-colour cosine similarity (MC dropout mean)


...

,0,45,90,135,180,225,270,315
0,1.0000,0.8742,0.8702,0.8414,0.8987,0.8388,0.8844,0.8669
45,0.8742,1.0000,0.8964,0.9174,0.8415,0.9260,0.9153,0.9753
90,0.8702,0.8964,1.0000,0.8717,0.8755,0.8744,0.9633,0.8836
135,0.8414,0.9174,0.8717,1.0000,0.8647,0.9555,0.8938,0.9167
180,0.8987,0.8415,0.8755,0.8647,1.0000,0.8484,0.8897,0.8301
225,0.8388,0.9260,0.8744,0.9555,0.8484,1.0000,0.8994,0.9228
270,0.8844,0.9153,0.9633,0.8938,0.8897,0.8994,1.0000,0.9130
315,0.8669,0.9753,0.8836,0.9167,0.8301,0.9228,0.9130,1.0000



Yellow — cross-angle cosine similarity (MC dropout mean)


,0,45,90,135,180,225,270,315
0,1.0000,0.9022,0.8898,0.8961,0.9225,0.8588,0.8987,0.8917
45,0.9022,1.0000,0.8781,0.9473,0.8656,0.9220,0.9062,0.9635
90,0.8898,0.8781,1.0000,0.8935,0.8708,0.8692,0.9496,0.8862
135,0.8961,0.9473,0.8935,1.0000,0.8872,0.9464,0.9153,0.9488
180,0.9225,0.8656,0.8708,0.8872,1.0000,0.8537,0.8763,0.8510
225,0.8588,0.9220,0.8692,0.9464,0.8537,1.0000,0.9001,0.9395
270,0.8987,0.9062,0.9496,0.9153,0.8763,0.9001,1.0000,0.9138
315,0.8917,0.9635,0.8862,0.9488,0.8510,0.9395,0.9138,1.0000



Wolf — cross-angle cosine similarity (MC dropout mean)


,0,45,90,135,180,225,270,315
0,1.0000,0.9147,0.8780,0.8835,0.9834,0.9113,0.8892,0.9129
45,0.9147,1.0000,0.9102,0.9605,0.8955,0.9628,0.8647,0.9764
90,0.8780,0.9102,1.0000,0.9015,0.8675,0.9089,0.9335,0.9093
135,0.8835,0.9605,0.9015,1.0000,0.8735,0.9632,0.8614,0.9581
180,0.9834,0.8955,0.8675,0.8735,1.0000,0.9004,0.8952,0.8932
225,0.9113,0.9628,0.9089,0.9632,0.9004,1.0000,0.8786,0.9719
270,0.8892,0.8647,0.9335,0.8614,0.8952,0.8786,1.0000,0.8759
315,0.9129,0.9764,0.9093,0.9581,0.8932,0.9719,0.8759,1.0000


## Gaussian noise variants (deterministic)

In [9]:
det_gaussian_results = {}
model_tag = MODEL_ID.replace('/', '_')

for sigma in GAUSSIAN_SIGMAS:
    sigma_tag = format_float_tag(sigma)
    variant_key = f'gaussian_sigma{sigma_tag}_rep{GAUSSIAN_REPEATS}'
    cache_path = CACHE_ROOT / f'{model_tag}_det_{variant_key}.pt'

    def builder(s=sigma):
        set_determinism(SEED)
        loaded = load_clip_backbone(MODEL_ID, device=DEVICE)
        model, processor, device = loaded.model, loaded.processor, loaded.device
        model.eval()

        def generator(inputs):
            with torch.no_grad():
                embedding = model.get_image_features(**inputs)
            embedding = normalise_embeddings(embedding).squeeze(0)
            return embedding

        transform = gaussian_transform_factory(s)
        return compute_variant_embeddings(
            dataset_df,
            processor,
            device,
            generator,
            transform_fn=transform,
            repeats=GAUSSIAN_REPEATS,
        )

    det_embeddings, created = ensure_cached_embeddings(cache_path, builder)
    if created:
        print(f'Computed deterministic Gaussian σ={sigma:.3f} embeddings and cached at {cache_path}')
    else:
        print(f'Loaded deterministic Gaussian σ={sigma:.3f} embeddings from {cache_path}')

    angle_tables, colour_tables = build_similarity_tables(det_embeddings, colours, angles, DISPLAY_NAMES)
    det_gaussian_results[sigma] = {'angle': angle_tables, 'colour': colour_tables}


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Computed deterministic Gaussian σ=0.010 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_gaussian_sigma0p01_rep16.pt
Computed deterministic Gaussian σ=0.020 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_gaussian_sigma0p02_rep16.pt
Computed deterministic Gaussian σ=0.040 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_gaussian_sigma0p04_rep16.pt
Computed deterministic Gaussian σ=0.080 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_gaussian_sigma0p08_rep16.pt
Computed deterministic Gaussian σ=0.160 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_gaussian_sigma0p16_rep16.pt
Computed deterministic Gaussian σ=0.320 

In [11]:
for sigma in GAUSSIAN_SIGMAS:
    results = det_gaussian_results[sigma]
    angle_tables = results['angle']
    colour_tables = results['colour']
    tables = list(angle_tables.values()) + list(colour_tables.values())
    if not tables:
        continue
    vmin = min(table.values.min() for table in tables)
    vmax = max(table.values.max() for table in tables)

    print(f"\nGaussian noise σ={sigma:.3f} — cross-colour cosine similarity (deterministic)")
    for angle in angles:
        display(
            angle_tables[angle]
            .style.format('{:.4f}')
            .background_gradient(cmap='viridis', vmin=vmin, vmax=vmax)
        )

    print('\nDeterministic cross-angle cosine similarity by colour:')
    for colour in colours:
        label = DISPLAY_NAMES.get(colour, colour)
        display(
            colour_tables[colour]
            .style.format('{:.4f}')
            .background_gradient(cmap='viridis', vmin=vmin, vmax=vmax)
        )



Gaussian noise σ=0.010 — cross-colour cosine similarity (deterministic)


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9060,0.8860,0.9257,0.6678
Magenta,0.9060,1.0000,0.8712,0.8876,0.6613
White,0.8860,0.8712,1.0000,0.9137,0.6735
Yellow,0.9257,0.8876,0.9137,1.0000,0.6873
Wolf,0.6678,0.6613,0.6735,0.6873,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9128,0.8980,0.9067,0.6890
Magenta,0.9128,1.0000,0.8776,0.8861,0.6659
White,0.8980,0.8776,1.0000,0.9297,0.7731
Yellow,0.9067,0.8861,0.9297,1.0000,0.7306
Wolf,0.6890,0.6659,0.7731,0.7306,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.8826,0.8327,0.8709,0.7266
Magenta,0.8826,1.0000,0.8252,0.8253,0.6996
White,0.8327,0.8252,1.0000,0.8729,0.7397
Yellow,0.8709,0.8253,0.8729,1.0000,0.7240
Wolf,0.7266,0.6996,0.7397,0.7240,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9293,0.9279,0.9153,0.6703
Magenta,0.9293,1.0000,0.8949,0.8978,0.6445
White,0.9279,0.8949,1.0000,0.9221,0.6824
Yellow,0.9153,0.8978,0.9221,1.0000,0.7223
Wolf,0.6703,0.6445,0.6824,0.7223,1.0000


...

,0,45,90,135,180,225,270,315
0,1.0000,0.9187,0.8631,0.9013,0.9157,0.9051,0.8629,0.9259
45,0.9187,1.0000,0.8916,0.9622,0.9050,0.9665,0.8904,0.9861
90,0.8631,0.8916,1.0000,0.8601,0.8991,0.8946,0.9859,0.9026
135,0.9013,0.9622,0.8601,1.0000,0.9104,0.9843,0.8526,0.9497
180,0.9157,0.9050,0.8991,0.9104,1.0000,0.9179,0.8838,0.9075
225,0.9051,0.9665,0.8946,0.9843,0.9179,1.0000,0.8912,0.9626
270,0.8629,0.8904,0.9859,0.8526,0.8838,0.8912,1.0000,0.9087
315,0.9259,0.9861,0.9026,0.9497,0.9075,0.9626,0.9087,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.8987,0.8633,0.8961,0.9082,0.8832,0.8581,0.9116
45,0.8987,1.0000,0.9251,0.9660,0.9065,0.9658,0.9232,0.9900
90,0.8633,0.9251,1.0000,0.9188,0.9196,0.9350,0.9884,0.9346
135,0.8961,0.9660,0.9188,1.0000,0.9285,0.9831,0.9102,0.9711
180,0.9082,0.9065,0.9196,0.9285,1.0000,0.9296,0.9108,0.9206
225,0.8832,0.9658,0.9350,0.9831,0.9296,1.0000,0.9299,0.9694
270,0.8581,0.9232,0.9884,0.9102,0.9108,0.9299,1.0000,0.9291
315,0.9116,0.9900,0.9346,0.9711,0.9206,0.9694,0.9291,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9161,0.8818,0.9123,0.9247,0.9069,0.8897,0.9116
45,0.9161,1.0000,0.9067,0.9555,0.8920,0.9581,0.9217,0.9785
90,0.8818,0.9067,1.0000,0.9082,0.8980,0.9246,0.9902,0.9263
135,0.9123,0.9555,0.9082,1.0000,0.9334,0.9823,0.9188,0.9517
180,0.9247,0.8920,0.8980,0.9334,1.0000,0.9270,0.8987,0.8959
225,0.9069,0.9581,0.9246,0.9823,0.9270,1.0000,0.9324,0.9642
270,0.8897,0.9217,0.9902,0.9188,0.8987,0.9324,1.0000,0.9414
315,0.9116,0.9785,0.9263,0.9517,0.8959,0.9642,0.9414,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9293,0.9015,0.9137,0.9327,0.9160,0.8961,0.9229
45,0.9293,1.0000,0.9219,0.9658,0.9030,0.9594,0.9329,0.9887
90,0.9015,0.9219,1.0000,0.9154,0.9012,0.9208,0.9818,0.9382
135,0.9137,0.9658,0.9154,1.0000,0.9263,0.9838,0.9175,0.9683
180,0.9327,0.9030,0.9012,0.9263,1.0000,0.9215,0.8840,0.9022
225,0.9160,0.9594,0.9208,0.9838,0.9215,1.0000,0.9197,0.9652
270,0.8961,0.9329,0.9818,0.9175,0.8840,0.9197,1.0000,0.9425
315,0.9229,0.9887,0.9382,0.9683,0.9022,0.9652,0.9425,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9189,0.8895,0.9021,0.9976,0.9413,0.9018,0.9647
45,0.9189,1.0000,0.9525,0.9591,0.9188,0.9386,0.8594,0.9362
90,0.8895,0.9525,1.0000,0.9511,0.8850,0.9298,0.8641,0.9052
135,0.9021,0.9591,0.9511,1.0000,0.8994,0.9432,0.8388,0.9075
180,0.9976,0.9188,0.8850,0.8994,1.0000,0.9369,0.8965,0.9631
225,0.9413,0.9386,0.9298,0.9432,0.9369,1.0000,0.9056,0.9485
270,0.9018,0.8594,0.8641,0.8388,0.8965,0.9056,1.0000,0.9330
315,0.9647,0.9362,0.9052,0.9075,0.9631,0.9485,0.9330,1.0000


## Gaussian noise variants (MC dropout)

In [12]:
mc_gaussian_results = {}
model_tag = MODEL_ID.replace('/', '_')

for sigma in GAUSSIAN_SIGMAS:
    sigma_tag = format_float_tag(sigma)
    variant_key = f'gaussian_sigma{sigma_tag}_rep{GAUSSIAN_REPEATS}_p{format_float_tag(MC_DROPOUT_P)}_passes{MC_PASSES}'
    cache_path = CACHE_ROOT / f'{model_tag}_mc_{variant_key}.pt'

    def builder(s=sigma):
        set_determinism(SEED)
        loaded_mc = load_clip_backbone(MODEL_ID, device=DEVICE)
        model_mc, processor_mc, device_mc = loaded_mc.model, loaded_mc.processor, loaded_mc.device
        model_mc.eval()
        insert_adapters(model_mc, ADAPTER_TARGETS, p=MC_DROPOUT_P)
        override_dropout_rate(model_mc, MC_DROPOUT_P)

        def mcdo_generator(inputs):
            samples = sample_embeddings(
                model=model_mc,
                forward_fn=model_mc.get_image_features,
                forward_kwargs=inputs,
                passes=MC_PASSES,
                microbatch=MC_MICROBATCH,
            )
            model_mc.eval()
            samples = normalise_embeddings(samples)
            mean_embed = samples.mean(dim=0, keepdim=True)
            mean_embed = normalise_embeddings(mean_embed).squeeze(0)
            return mean_embed

        transform = gaussian_transform_factory(s)
        return compute_variant_embeddings(
            dataset_df,
            processor_mc,
            device_mc,
            mcdo_generator,
            transform_fn=transform,
            repeats=GAUSSIAN_REPEATS,
        )

    mc_embeddings, created = ensure_cached_embeddings(cache_path, builder)
    if created:
        print(f'Computed MC dropout Gaussian σ={sigma:.3f} embeddings and cached at {cache_path}')
    else:
        print(f'Loaded MC dropout Gaussian σ={sigma:.3f} embeddings from {cache_path}')

    angle_tables, colour_tables = build_similarity_tables(mc_embeddings, colours, angles, DISPLAY_NAMES)
    mc_gaussian_results[sigma] = {'angle': angle_tables, 'colour': colour_tables}


Computed MC dropout Gaussian σ=0.010 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_gaussian_sigma0p01_rep16_p0p01_passes32.pt
Computed MC dropout Gaussian σ=0.020 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_gaussian_sigma0p02_rep16_p0p01_passes32.pt
Computed MC dropout Gaussian σ=0.040 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_gaussian_sigma0p04_rep16_p0p01_passes32.pt
Computed MC dropout Gaussian σ=0.080 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_gaussian_sigma0p08_rep16_p0p01_passes32.pt
Computed MC dropout Gaussian σ=0.160 embeddings and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_gaussian_sigma0p16_rep16_p0p

In [13]:
for sigma in GAUSSIAN_SIGMAS:
    results = mc_gaussian_results[sigma]
    angle_tables = results['angle']
    colour_tables = results['colour']
    tables = list(angle_tables.values()) + list(colour_tables.values())
    if not tables:
        continue
    vmin = min(table.values.min() for table in tables)
    vmax = max(table.values.max() for table in tables)

    print(f"\nGaussian noise σ={sigma:.3f} — cross-colour cosine similarity (MC dropout mean)")
    for angle in angles:
        display(
            angle_tables[angle]
            .style.format('{:.4f}')
            .background_gradient(cmap='viridis', vmin=vmin, vmax=vmax)
        )

    print('\nMC dropout cross-angle cosine similarity by colour:')
    for colour in colours:
        label = DISPLAY_NAMES.get(colour, colour)
        display(
            colour_tables[colour]
            .style.format('{:.4f}')
            .background_gradient(cmap='viridis', vmin=vmin, vmax=vmax)
        )



Gaussian noise σ=0.010 — cross-colour cosine similarity (MC dropout mean)


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9195,0.9111,0.9357,0.7286
Magenta,0.9195,1.0000,0.8860,0.9014,0.7187
White,0.9111,0.8860,1.0000,0.9296,0.7471
Yellow,0.9357,0.9014,0.9296,1.0000,0.7525
Wolf,0.7286,0.7187,0.7471,0.7525,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9252,0.9170,0.9225,0.7403
Magenta,0.9252,1.0000,0.8929,0.9039,0.7137
White,0.9170,0.8929,1.0000,0.9409,0.8021
Yellow,0.9225,0.9039,0.9409,1.0000,0.7661
Wolf,0.7403,0.7137,0.8021,0.7661,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9178,0.8818,0.9008,0.7725
Magenta,0.9178,1.0000,0.8609,0.8639,0.7413
White,0.8818,0.8609,1.0000,0.9049,0.7788
Yellow,0.9008,0.8639,0.9049,1.0000,0.7673
Wolf,0.7725,0.7413,0.7788,0.7673,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9351,0.9385,0.9229,0.7091
Magenta,0.9351,1.0000,0.9052,0.9111,0.6915
White,0.9385,0.9052,1.0000,0.9314,0.7229
Yellow,0.9229,0.9111,0.9314,1.0000,0.7542
Wolf,0.7091,0.6915,0.7229,0.7542,1.0000


...

,0,45,90,135,180,225,270,315
0,1.0000,0.9344,0.8878,0.9215,0.9356,0.9232,0.8810,0.9397
45,0.9344,1.0000,0.9143,0.9703,0.9225,0.9730,0.9070,0.9895
90,0.8878,0.9143,1.0000,0.8868,0.9170,0.9136,0.9898,0.9262
135,0.9215,0.9703,0.8868,1.0000,0.9303,0.9884,0.8742,0.9616
180,0.9356,0.9225,0.9170,0.9303,1.0000,0.9358,0.8999,0.9259
225,0.9232,0.9730,0.9136,0.9884,0.9358,1.0000,0.9035,0.9709
270,0.8810,0.9070,0.9898,0.8742,0.8999,0.9035,1.0000,0.9237
315,0.9397,0.9895,0.9262,0.9616,0.9259,0.9709,0.9237,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9198,0.8939,0.9169,0.9359,0.9112,0.8881,0.9290
45,0.9198,1.0000,0.9417,0.9724,0.9283,0.9743,0.9397,0.9920
90,0.8939,0.9417,1.0000,0.9341,0.9329,0.9442,0.9902,0.9488
135,0.9169,0.9724,0.9341,1.0000,0.9415,0.9891,0.9258,0.9773
180,0.9359,0.9283,0.9329,0.9415,1.0000,0.9414,0.9262,0.9377
225,0.9112,0.9743,0.9442,0.9891,0.9414,1.0000,0.9399,0.9786
270,0.8881,0.9397,0.9902,0.9258,0.9262,0.9399,1.0000,0.9441
315,0.9290,0.9920,0.9488,0.9773,0.9377,0.9786,0.9441,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9394,0.9072,0.9340,0.9415,0.9349,0.9166,0.9379
45,0.9394,1.0000,0.9285,0.9670,0.9155,0.9701,0.9417,0.9870
90,0.9072,0.9285,1.0000,0.9275,0.9152,0.9393,0.9927,0.9433
135,0.9340,0.9670,0.9275,1.0000,0.9482,0.9898,0.9385,0.9641
180,0.9415,0.9155,0.9152,0.9482,1.0000,0.9453,0.9194,0.9186
225,0.9349,0.9701,0.9393,0.9898,0.9453,1.0000,0.9485,0.9726
270,0.9166,0.9417,0.9927,0.9385,0.9194,0.9485,1.0000,0.9556
315,0.9379,0.9870,0.9433,0.9641,0.9186,0.9726,0.9556,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9435,0.9177,0.9313,0.9451,0.9324,0.9148,0.9383
45,0.9435,1.0000,0.9338,0.9728,0.9195,0.9702,0.9433,0.9919
90,0.9177,0.9338,1.0000,0.9292,0.9170,0.9318,0.9875,0.9514
135,0.9313,0.9728,0.9292,1.0000,0.9418,0.9901,0.9316,0.9735
180,0.9451,0.9195,0.9170,0.9418,1.0000,0.9379,0.9078,0.9196
225,0.9324,0.9702,0.9318,0.9901,0.9379,1.0000,0.9345,0.9723
270,0.9148,0.9433,0.9875,0.9316,0.9078,0.9345,1.0000,0.9556
315,0.9383,0.9919,0.9514,0.9735,0.9196,0.9723,0.9556,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9352,0.9101,0.9110,0.9982,0.9523,0.9183,0.9696
45,0.9352,1.0000,0.9632,0.9676,0.9361,0.9552,0.8895,0.9494
90,0.9101,0.9632,1.0000,0.9602,0.9069,0.9480,0.9017,0.9287
135,0.9110,0.9676,0.9602,1.0000,0.9095,0.9516,0.8619,0.9172
180,0.9982,0.9361,0.9069,0.9095,1.0000,0.9501,0.9152,0.9693
225,0.9523,0.9552,0.9480,0.9516,0.9501,1.0000,0.9240,0.9631
270,0.9183,0.8895,0.9017,0.8619,0.9152,0.9240,1.0000,0.9488
315,0.9696,0.9494,0.9287,0.9172,0.9693,0.9631,0.9488,1.0000


## Downsampled variants (deterministic)

In [14]:
downsample_specs = {
    'pixel': {
        'label': 'Pixelated (nearest)',
        'factory': pixelate_transform,
    },
    'bicubic': {
        'label': 'Bicubic (smoothed)',
        'factory': bicubic_transform,
    },
}

det_downsample_results = {}
model_tag = MODEL_ID.replace('/', '_')

for method_key, spec in downsample_specs.items():
    det_downsample_results[method_key] = {}
    for percent in DOWNSAMPLE_PERCENTS:
        variant_key = f'{method_key}_{percent:02d}pct'
        cache_path = CACHE_ROOT / f'{model_tag}_det_{variant_key}.pt'

        def builder(method=spec['factory'], pct=percent):
            set_determinism(SEED)
            loaded = load_clip_backbone(MODEL_ID, device=DEVICE)
            model, processor, device = loaded.model, loaded.processor, loaded.device
            model.eval()

            def generator(inputs):
                with torch.no_grad():
                    embedding = model.get_image_features(**inputs)
                embedding = normalise_embeddings(embedding).squeeze(0)
                return embedding

            transform = method(pct)
            return compute_variant_embeddings(
                dataset_df,
                processor,
                device,
                generator,
                transform_fn=transform,
                repeats=1,
            )

        embeddings, created = ensure_cached_embeddings(cache_path, builder)
        if created:
            print(f"Computed deterministic {spec['label']} embeddings at −{percent}% and cached at {cache_path}")
        else:
            print(f"Loaded deterministic {spec['label']} embeddings at −{percent}% from {cache_path}")

        angle_tables, colour_tables = build_similarity_tables(embeddings, colours, angles, DISPLAY_NAMES)
        det_downsample_results[method_key][percent] = {'angle': angle_tables, 'colour': colour_tables, 'label': spec['label']}


Computed deterministic Pixelated (nearest) embeddings at −10% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_pixel_10pct.pt
Computed deterministic Pixelated (nearest) embeddings at −20% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_pixel_20pct.pt
Computed deterministic Pixelated (nearest) embeddings at −40% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_pixel_40pct.pt
Computed deterministic Pixelated (nearest) embeddings at −60% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_pixel_60pct.pt
Computed deterministic Pixelated (nearest) embeddings at −80% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_det_pixel_80pct.pt
Computed deterministic Bicubic (smoothed) embeddin

In [15]:
for method_key, method_results in det_downsample_results.items():
    method_label = next(iter(method_results.values()))['label'] if method_results else method_key
    for percent in DOWNSAMPLE_PERCENTS:
        results = method_results.get(percent)
        if results is None:
            continue
        angle_tables = results['angle']
        colour_tables = results['colour']
        tables = list(angle_tables.values()) + list(colour_tables.values())
        if not tables:
            continue
        vmin = min(table.values.min() for table in tables)
        vmax = max(table.values.max() for table in tables)

        print(f"\n{method_label} downsample −{percent}% — cross-colour cosine similarity (deterministic)")
        for angle in angles:
            display(
                angle_tables[angle]
                .style.format('{:.4f}')
                .background_gradient(cmap='viridis', vmin=vmin, vmax=vmax)
            )

        print('\nDeterministic cross-angle cosine similarity by colour:')
        for colour in colours:
            label = DISPLAY_NAMES.get(colour, colour)
            display(
                colour_tables[colour]
                .style.format('{:.4f}')
                .background_gradient(cmap='viridis', vmin=vmin, vmax=vmax)
            )



Pixelated (nearest) downsample −10% — cross-colour cosine similarity (deterministic)


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.8809,0.8714,0.8992,0.6889
Magenta,0.8809,1.0000,0.8730,0.8834,0.6622
White,0.8714,0.8730,1.0000,0.9191,0.7073
Yellow,0.8992,0.8834,0.9191,1.0000,0.7146
Wolf,0.6889,0.6622,0.7073,0.7146,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9119,0.8843,0.9068,0.6855
Magenta,0.9119,1.0000,0.8748,0.8881,0.6714
White,0.8843,0.8748,1.0000,0.9183,0.7744
Yellow,0.9068,0.8881,0.9183,1.0000,0.7356
Wolf,0.6855,0.6714,0.7744,0.7356,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.8840,0.8826,0.8631,0.7423
Magenta,0.8840,1.0000,0.8583,0.8252,0.7029
White,0.8826,0.8583,1.0000,0.8966,0.7772
Yellow,0.8631,0.8252,0.8966,1.0000,0.7454
Wolf,0.7423,0.7029,0.7772,0.7454,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9178,0.9028,0.9049,0.7020
Magenta,0.9178,1.0000,0.8939,0.8844,0.6531
White,0.9028,0.8939,1.0000,0.9221,0.7035
Yellow,0.9049,0.8844,0.9221,1.0000,0.7379
Wolf,0.7020,0.6531,0.7035,0.7379,1.0000


...

,0,45,90,135,180,225,270,315
0,1.0000,0.8964,0.8703,0.8822,0.9241,0.8830,0.9022,0.9017
45,0.8964,1.0000,0.8484,0.9353,0.8921,0.9299,0.9244,0.9711
90,0.8703,0.8484,1.0000,0.8232,0.8758,0.8352,0.9256,0.8691
135,0.8822,0.9353,0.8232,1.0000,0.9016,0.9629,0.9037,0.9307
180,0.9241,0.8921,0.8758,0.9016,1.0000,0.8961,0.8985,0.8998
225,0.8830,0.9299,0.8352,0.9629,0.8961,1.0000,0.9052,0.9355
270,0.9022,0.9244,0.9256,0.9037,0.8985,0.9052,1.0000,0.9448
315,0.9017,0.9711,0.8691,0.9307,0.8998,0.9355,0.9448,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.8972,0.8967,0.8807,0.9223,0.8989,0.8869,0.8961
45,0.8972,1.0000,0.9307,0.9395,0.9144,0.9644,0.9147,0.9785
90,0.8967,0.9307,1.0000,0.8939,0.9207,0.9261,0.9391,0.9165
135,0.8807,0.9395,0.8939,1.0000,0.8910,0.9704,0.9181,0.9400
180,0.9223,0.9144,0.9207,0.8910,1.0000,0.9126,0.8751,0.8986
225,0.8989,0.9644,0.9261,0.9704,0.9126,1.0000,0.9367,0.9624
270,0.8869,0.9147,0.9391,0.9181,0.8751,0.9367,1.0000,0.9273
315,0.8961,0.9785,0.9165,0.9400,0.8986,0.9624,0.9273,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.8648,0.8516,0.8796,0.8854,0.8785,0.8692,0.8543
45,0.8648,1.0000,0.8714,0.9047,0.8222,0.9213,0.8805,0.9641
90,0.8516,0.8714,1.0000,0.8489,0.8486,0.8346,0.9623,0.8516
135,0.8796,0.9047,0.8489,1.0000,0.8799,0.9778,0.8801,0.8959
180,0.8854,0.8222,0.8486,0.8799,1.0000,0.8709,0.8640,0.8258
225,0.8785,0.9213,0.8346,0.9778,0.8709,1.0000,0.8630,0.9168
270,0.8692,0.8805,0.9623,0.8801,0.8640,0.8630,1.0000,0.8713
315,0.8543,0.9641,0.8516,0.8959,0.8258,0.9168,0.8713,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.8779,0.9004,0.8880,0.9123,0.8760,0.8852,0.8841
45,0.8779,1.0000,0.8986,0.9464,0.8647,0.9485,0.8973,0.9676
90,0.9004,0.8986,1.0000,0.9019,0.8827,0.9005,0.9580,0.9044
135,0.8880,0.9464,0.9019,1.0000,0.8981,0.9668,0.9059,0.9455
180,0.9123,0.8647,0.8827,0.8981,1.0000,0.8659,0.8725,0.8623
225,0.8760,0.9485,0.9005,0.9668,0.8659,1.0000,0.9018,0.9579
270,0.8852,0.8973,0.9580,0.9059,0.8725,0.9018,1.0000,0.9021
315,0.8841,0.9676,0.9044,0.9455,0.8623,0.9579,0.9021,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.8447,0.8126,0.8325,0.9854,0.8414,0.7829,0.8254
45,0.8447,1.0000,0.9364,0.9635,0.8451,0.9557,0.7581,0.9711
90,0.8126,0.9364,1.0000,0.9334,0.8093,0.9211,0.8080,0.9417
135,0.8325,0.9635,0.9334,1.0000,0.8335,0.9493,0.7718,0.9635
180,0.9854,0.8451,0.8093,0.8335,1.0000,0.8382,0.7644,0.8217
225,0.8414,0.9557,0.9211,0.9493,0.8382,1.0000,0.7513,0.9424
270,0.7829,0.7581,0.8080,0.7718,0.7644,0.7513,1.0000,0.7749
315,0.8254,0.9711,0.9417,0.9635,0.8217,0.9424,0.7749,1.0000


## Downsampled variants (MC dropout)

In [16]:
mc_downsample_results = {}
model_tag = MODEL_ID.replace('/', '_')

for method_key, spec in downsample_specs.items():
    mc_downsample_results[method_key] = {}
    for percent in DOWNSAMPLE_PERCENTS:
        variant_key = f'{method_key}_{percent:02d}pct_p{format_float_tag(MC_DROPOUT_P)}_passes{MC_PASSES}'
        cache_path = CACHE_ROOT / f'{model_tag}_mc_{variant_key}.pt'

        def builder(method=spec['factory'], pct=percent):
            set_determinism(SEED)
            loaded_mc = load_clip_backbone(MODEL_ID, device=DEVICE)
            model_mc, processor_mc, device_mc = loaded_mc.model, loaded_mc.processor, loaded_mc.device
            model_mc.eval()
            insert_adapters(model_mc, ADAPTER_TARGETS, p=MC_DROPOUT_P)
            override_dropout_rate(model_mc, MC_DROPOUT_P)

            def mcdo_generator(inputs):
                samples = sample_embeddings(
                    model=model_mc,
                    forward_fn=model_mc.get_image_features,
                    forward_kwargs=inputs,
                    passes=MC_PASSES,
                    microbatch=MC_MICROBATCH,
                )
                model_mc.eval()
                samples = normalise_embeddings(samples)
                mean_embed = samples.mean(dim=0, keepdim=True)
                mean_embed = normalise_embeddings(mean_embed).squeeze(0)
                return mean_embed

            transform = method(pct)
            return compute_variant_embeddings(
                dataset_df,
                processor_mc,
                device_mc,
                mcdo_generator,
                transform_fn=transform,
                repeats=1,
            )

        embeddings, created = ensure_cached_embeddings(cache_path, builder)
        if created:
            print(f"Computed MC dropout {spec['label']} embeddings at −{percent}% and cached at {cache_path}")
        else:
            print(f"Loaded MC dropout {spec['label']} embeddings at −{percent}% from {cache_path}")

        angle_tables, colour_tables = build_similarity_tables(embeddings, colours, angles, DISPLAY_NAMES)
        mc_downsample_results[method_key][percent] = {'angle': angle_tables, 'colour': colour_tables, 'label': spec['label']}


Computed MC dropout Pixelated (nearest) embeddings at −10% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_pixel_10pct_p0p01_passes32.pt
Computed MC dropout Pixelated (nearest) embeddings at −20% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_pixel_20pct_p0p01_passes32.pt
Computed MC dropout Pixelated (nearest) embeddings at −40% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_pixel_40pct_p0p01_passes32.pt
Computed MC dropout Pixelated (nearest) embeddings at −60% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_pixel_60pct_p0p01_passes32.pt
Computed MC dropout Pixelated (nearest) embeddings at −80% and cached at /home/fwromano/Documents/Code/UCLIP/runs/sim2_angle_colour_cosine/openai_clip-vit-base-patch32_mc_pixel_80pct_p0p01_passes3

In [17]:
for method_key, method_results in mc_downsample_results.items():
    method_label = next(iter(method_results.values()))['label'] if method_results else method_key
    for percent in DOWNSAMPLE_PERCENTS:
        results = method_results.get(percent)
        if results is None:
            continue
        angle_tables = results['angle']
        colour_tables = results['colour']
        tables = list(angle_tables.values()) + list(colour_tables.values())
        if not tables:
            continue
        vmin = min(table.values.min() for table in tables)
        vmax = max(table.values.max() for table in tables)

        print(f"\n{method_label} downsample −{percent}% — cross-colour cosine similarity (MC dropout mean)")
        for angle in angles:
            display(
                angle_tables[angle]
                .style.format('{:.4f}')
                .background_gradient(cmap='viridis', vmin=vmin, vmax=vmax)
            )

        print('\nMC dropout cross-angle cosine similarity by colour:')
        for colour in colours:
            label = DISPLAY_NAMES.get(colour, colour)
            display(
                colour_tables[colour]
                .style.format('{:.4f}')
                .background_gradient(cmap='viridis', vmin=vmin, vmax=vmax)
            )



Pixelated (nearest) downsample −10% — cross-colour cosine similarity (MC dropout mean)


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9065,0.9045,0.9289,0.7464
Magenta,0.9065,1.0000,0.8802,0.8916,0.6977
White,0.9045,0.8802,1.0000,0.9273,0.7469
Yellow,0.9289,0.8916,0.9273,1.0000,0.7640
Wolf,0.7464,0.6977,0.7469,0.7640,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9168,0.9048,0.9181,0.7411
Magenta,0.9168,1.0000,0.8854,0.8926,0.7183
White,0.9048,0.8854,1.0000,0.9289,0.8150
Yellow,0.9181,0.8926,0.9289,1.0000,0.7653
Wolf,0.7411,0.7183,0.8150,0.7653,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9186,0.9122,0.8969,0.7904
Magenta,0.9186,1.0000,0.8833,0.8625,0.7297
White,0.9122,0.8833,1.0000,0.9215,0.8004
Yellow,0.8969,0.8625,0.9215,1.0000,0.7904
Wolf,0.7904,0.7297,0.8004,0.7904,1.0000


,Indigo,Magenta,White,Yellow,Wolf
Indigo,1.0000,0.9259,0.9145,0.9161,0.7267
Magenta,0.9259,1.0000,0.9010,0.9029,0.6947
White,0.9145,0.9010,1.0000,0.9282,0.7314
Yellow,0.9161,0.9029,0.9282,1.0000,0.7735
Wolf,0.7267,0.6947,0.7314,0.7735,1.0000


...

,0,45,90,135,180,225,270,315
0,1.0000,0.9127,0.9075,0.9004,0.9426,0.9003,0.9310,0.9205
45,0.9127,1.0000,0.8801,0.9526,0.9084,0.9470,0.9383,0.9776
90,0.9075,0.8801,1.0000,0.8647,0.9197,0.8836,0.9519,0.9033
135,0.9004,0.9526,0.8647,1.0000,0.9164,0.9703,0.9234,0.9494
180,0.9426,0.9084,0.9197,0.9164,1.0000,0.9175,0.9313,0.9141
225,0.9003,0.9470,0.8836,0.9703,0.9175,1.0000,0.9304,0.9559
270,0.9310,0.9383,0.9519,0.9234,0.9313,0.9304,1.0000,0.9574
315,0.9205,0.9776,0.9033,0.9494,0.9141,0.9559,0.9574,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9236,0.9220,0.9058,0.9384,0.9146,0.9171,0.9171
45,0.9236,1.0000,0.9509,0.9626,0.9357,0.9733,0.9593,0.9824
90,0.9220,0.9509,1.0000,0.9238,0.9423,0.9433,0.9686,0.9423
135,0.9058,0.9626,0.9238,1.0000,0.9171,0.9814,0.9447,0.9579
180,0.9384,0.9357,0.9423,0.9171,1.0000,0.9271,0.9275,0.9228
225,0.9146,0.9733,0.9433,0.9814,0.9271,1.0000,0.9614,0.9722
270,0.9171,0.9593,0.9686,0.9447,0.9275,0.9614,1.0000,0.9566
315,0.9171,0.9824,0.9423,0.9579,0.9228,0.9722,0.9566,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9043,0.8893,0.9042,0.9150,0.9008,0.9025,0.8884
45,0.9043,1.0000,0.9184,0.9296,0.8773,0.9426,0.9227,0.9702
90,0.8893,0.9184,1.0000,0.8908,0.8875,0.8825,0.9717,0.8917
135,0.9042,0.9296,0.8908,1.0000,0.9164,0.9807,0.9125,0.9142
180,0.9150,0.8773,0.8875,0.9164,1.0000,0.9052,0.8944,0.8694
225,0.9008,0.9426,0.8825,0.9807,0.9052,1.0000,0.9044,0.9320
270,0.9025,0.9227,0.9717,0.9125,0.8944,0.9044,1.0000,0.9051
315,0.8884,0.9702,0.8917,0.9142,0.8694,0.9320,0.9051,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.9118,0.9272,0.9262,0.9354,0.9158,0.9172,0.9116
45,0.9118,1.0000,0.9276,0.9530,0.8988,0.9628,0.9267,0.9753
90,0.9272,0.9276,1.0000,0.9408,0.9259,0.9323,0.9741,0.9344
135,0.9262,0.9530,0.9408,1.0000,0.9323,0.9775,0.9416,0.9539
180,0.9354,0.8988,0.9259,0.9323,1.0000,0.9146,0.9165,0.9010
225,0.9158,0.9628,0.9323,0.9775,0.9146,1.0000,0.9320,0.9660
270,0.9172,0.9267,0.9741,0.9416,0.9165,0.9320,1.0000,0.9282
315,0.9116,0.9753,0.9344,0.9539,0.9010,0.9660,0.9282,1.0000


,0,45,90,135,180,225,270,315
0,1.0000,0.8881,0.8669,0.8677,0.9897,0.8946,0.8464,0.8670
45,0.8881,1.0000,0.9437,0.9732,0.8868,0.9685,0.8058,0.9779
90,0.8669,0.9437,1.0000,0.9411,0.8642,0.9374,0.8365,0.9459
135,0.8677,0.9732,0.9411,1.0000,0.8661,0.9623,0.8049,0.9710
180,0.9897,0.8868,0.8642,0.8661,1.0000,0.8936,0.8385,0.8656
225,0.8946,0.9685,0.9374,0.9623,0.8936,1.0000,0.8124,0.9659
270,0.8464,0.8058,0.8365,0.8049,0.8385,0.8124,1.0000,0.8025
315,0.8670,0.9779,0.9459,0.9710,0.8656,0.9659,0.8025,1.0000
